In [ ]:
import sys, platform, os
print("Python:", sys.version)
print("Platform:", platform.platform())
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))


In [ ]:
import torch
print("torch available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    total_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print("GPU memory (GB):", round(total_gb, 2))


In [ ]:
!pip -q install --upgrade pip
!pip -q install "torch==2.2.0" "torchvision==0.17.0" "torchaudio==2.2.0"
!pip -q install "transformers==4.40.1" "tokenizers==0.19.1" "timm==0.9.10"
!pip -q install pillow sentencepiece==0.1.99 accelerate einops bitsandbytes
print("Install step finished.")


In [ ]:
%cd /content
!rm -rf /content/openvla
!git clone https://github.com/openvla/openvla.git
print("Repo cloned.")


In [ ]:
from pathlib import Path

p = Path("/content/openvla/experiments/robot/openvla_utils.py")
txt = p.read_text()

txt = txt.replace(
    'print("[*] Loading in BF16 with Flash-Attention Enabled")',
    'print("[*] Loading model with eager attention fallback")'
)

txt = txt.replace(
    'attn_implementation="flash_attention_2",',
    'attn_implementation="eager",'
)

p.write_text(txt)
print("Patched openvla_utils.py")


In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig
from PIL import Image
import numpy as np
import torch

checkpoint = "openvla/openvla-7b-finetuned-libero-spatial"

print("Loading processor...")
processor = AutoProcessor.from_pretrained(checkpoint, trust_remote_code=True)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading model in 4-bit...")
model = AutoModelForVision2Seq.from_pretrained(
    checkpoint,
    attn_implementation="eager",
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True
)

print("Making dummy image...")
img = Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8)).convert("RGB")

prompt = "In: What action should the robot take to pick up the black bowl and place it on the plate?\nOut:"

print("Processing inputs...")
inputs = processor(prompt, img, return_tensors="pt")

for k, v in inputs.items():
    if hasattr(v, "to"):
        inputs[k] = v.to(model.device)

print("input_ids shape:", inputs["input_ids"].shape)
print("attention_mask shape:", inputs["attention_mask"].shape)
print("pixel_values shape:", inputs["pixel_values"].shape)

print("Calling predict_action...")
action = model.predict_action(**inputs, unnorm_key="libero_spatial", do_sample=False)
print("Predicted action:", action)


In [ ]:
# OPTIONAL: run only after the smoke test succeeds

%cd /content
!rm -rf /content/LIBERO
!git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git
!pip -q install -e /content/LIBERO --no-deps
!pip -q install "imageio[ffmpeg]" robosuite==1.4.1 bddl easydict cloudpickle gym
print("LIBERO install finished.")


In [ ]:
# OPTIONAL: run only after LIBERO install succeeds

import sys
sys.path.append("/content/LIBERO")
import libero
from libero.libero import benchmark
print("LIBERO import works.")
